# [5.3] Mamba from Scratch - Solutions

These cells validate the reference implementation in `solutions.py`. The source file is the durable implementation used by tests and report generation; this notebook keeps the same ARENA-style progression and shows where each claim is checked.

<img src="../../instructions/assets/mamba_scan_contract.svg" width="820">


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter5_modern_architectures"
section = "part3_mamba_from_scratch"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

from part3_mamba_from_scratch import solutions, tests


## 1. Discretization and Recurrent Scan

<details><summary>Expected output</summary>

```text
All tests in `test_discretize_selective_scan_shapes_and_stability` passed!
All tests in `test_selective_scan_single_step_manual` passed!
All tests in `test_recurrent_scan_matches_reference` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The reference path expands shared `B/C` tensors across `d_inner`, uses `A = -exp(A_log)` for stable decays, loops explicitly over sequence positions, and applies `D` plus `silu(z)` only in the output readout.

</details>


In [ ]:
tests.test_discretize_selective_scan_shapes_and_stability(solutions.discretize_selective_scan)
tests.test_selective_scan_single_step_manual(solutions.selective_scan_recurrent)
tests.test_recurrent_scan_matches_reference(solutions.selective_scan_recurrent)


## 2. Associative and Chunked Equivalence

<details><summary>Expected output</summary>

```text
All tests in `test_selective_scan_equivalence` passed!
All tests in `test_chunked_scan_equivalence` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The associative scan composes `(a, b)` transforms as `(a2*a1, a2*b1+b2)`. The chunked scan simply threads the final recurrent state from one chunk into the next.

</details>


In [ ]:
tests.test_selective_scan_equivalence(solutions.selective_scan_equivalence_smoke_test)
tests.test_chunked_scan_equivalence(solutions.chunked_scan_equivalence_smoke_test)


## 3. Tiny Block Parity

<details><summary>Expected output</summary>

```text
All tests in `test_tiny_mamba_block_matches_reference` passed!
All tests in `test_block_step_equivalence` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The full forward path left-pads for causal convolution, then scans the whole sequence. The step path keeps a rolling convolution window and calls the scan for one token using the previous SSM state.

</details>


In [ ]:
tests.test_tiny_mamba_block_matches_reference(solutions.TinyMambaBlock)
tests.test_block_step_equivalence(solutions.block_step_equivalence_smoke_test)


## 4. Tiny LM Cache Parity

<details><summary>Expected output</summary>

```text
All tests in `test_tiny_lm_matches_reference` passed!
All tests in `test_tiny_lm_cache_parity` passed!
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Help - solution idea</summary>

The LM wrapper stores one recurrent state per layer and ties the LM head to the token embedding matrix. Greedy generation feeds only the newest token after the first cached call.

</details>


In [ ]:
tests.test_tiny_lm_matches_reference(solutions.TinyMambaForCausalLM)
tests.test_tiny_lm_cache_parity(solutions.tiny_lm_cache_parity_smoke_test)
tests.test_notebook_contract(solutions.run_smoke_test)


## Signature Result

| Check | Current result | Acceptance rule |
|---|---:|---|
| Recurrent vs associative scan max diff | `4.77e-7` | `<= 1e-6` |
| Tiny LM cache max diff | `1.19e-6` | `<= 1e-5` |
| Official Mamba preflight | `true` | must pass |
| Fast kernels available | `true` | must pass |
| Batched/single top-1 agreement | `1.0` | exactly `1.0` |
| Generated new tokens | `4` | exactly `4` |
| Peak VRAM | `0.279 GB` | below `24 GB` |

<details><summary>Interpreting the signature result</summary>

The report proves execution-path equivalence for the didactic implementation and a real CUDA preflight for the official checkpoint. It does not claim full weight mapping from Mamba-130M-HF into the tiny course model.

</details>

<details><summary>Help - checking the committed report</summary>

The cell below reads `verification_report.json`; it is a report-backed check. Use the report generator to rerun the actual CUDA preflight when implementation or report inputs change.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    return {
        "device": gpu["device"],
        "torch": gpu["torch_version"],
        "cuda": gpu["cuda_version"],
        "scan_max_abs_diff": gpu["scan_max_abs_diff"],
        "cache_max_abs_diff": gpu["cache_max_abs_diff"],
        "official_preflight": gpu["official_mamba_logits_generation_preflight_passed"],
        "fast_kernel_available": gpu["official_mamba_fast_kernel_available"],
        "batched_single_top1_agreement": gpu["official_mamba_batched_single_top1_agreement"],
        "generated_new_tokens": gpu["official_mamba_generation_new_tokens"],
        "peak_vram_gb": gpu["peak_vram_gb"],
    }


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


run_gpu_test()


## Limitations

The reference solution is a small implementation and verification target. It does not train Mamba, claim long-context benchmark throughput, or prove full checkpoint weight-level parity. Its purpose is to make the selective scan, chunking, convolution cache, SSM cache, and official-checkpoint runtime evidence inspectable.
